# 02 – Ba baseline kiểm soát nhiễu cho nhận dạng phương ngữ 63 tỉnh

| Baseline | Trả lời câu hỏi |
|---|---|
| **A. Âm học thô**: MFCC-stats, LTAS (phổ trung bình dài hạn) | Mô hình có đang học *kênh thu âm / micro* thay vì phương ngữ không? |
| **B. Chỉ văn bản**: TF-IDF + LinearSVC, PhoBERT | Bao nhiêu thông tin nằm ở *từ vựng* trong transcript? |
| **C. XLS-R đóng băng**: probe từng tầng, weighted-sum + linear/MLP | Mức trần khi dùng đặc trưng self-supervised trên toàn bộ dữ liệu |

- **Runtime:** GPU **L4** (A100 cũng được).
- **Yêu cầu:** đã chạy xong notebook 01.
- **Thời gian:** khoảng 1–1,5 giờ. Đặc trưng được cache trên Drive, nên nếu bị ngắt chỉ cần *Run all* lại.
- **Kết quả:** `MyDrive/vimd_dl/results/baselines_report.json`. Gửi file này cho Claude.

In [ ]:
# ==== CẤU HÌNH ====
DRIVE_ROOT = "/content/drive/MyDrive/vimd_dl"
RUN = dict(acoustic=True, text_tfidf=True, phobert=True, xlsr=True)
PROVINCE_COL, TEXT_COL = "province_code", "text"
XLSR_MODEL = "facebook/wav2vec2-xls-r-300m"
XLSR_MAX_SEC, XLSR_BATCH = 12.0, 16
PHOBERT_MODEL = "vinai/phobert-base-v2"
PHOBERT_EPOCHS, PHOBERT_LR, PHOBERT_BS, PHOBERT_MAX_LEN = 4, 3e-5, 32, 192
WORKERS = None
SEED = 42

In [ ]:
import os, io, sys, glob, json, time, shutil, subprocess, warnings, multiprocessing as mp
import numpy as np, pandas as pd, pyarrow.parquet as pq, soundfile as sf, librosa
from sklearn.metrics import f1_score, top_k_accuracy_score
warnings.filterwarnings("ignore")
T0 = time.time()

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
if RUN["phobert"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyvi"], check=False)

LOCAL = "/content" if os.path.isdir("/content") else os.path.abspath("_work")
LOCAL_DATA, FEAT_DIR, RES_DIR = f"{LOCAL}/vimd16k", f"{DRIVE_ROOT}/features", f"{DRIVE_ROOT}/results"
for d in (LOCAL_DATA, FEAT_DIR, RES_DIR):
    os.makedirs(d, exist_ok=True)
WORKERS = WORKERS or os.cpu_count()

# Chép dữ liệu từ Drive về ổ local (đọc nhanh hơn nhiều)
t = time.time()
for f in sorted(glob.glob(f"{DRIVE_ROOT}/data16k/*.parquet")):
    dst = f"{LOCAL_DATA}/{os.path.basename(f)}"
    if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(f):
        shutil.copy(f, dst)
print(f"Dữ liệu local sẵn sàng sau {time.time() - t:.0f}s")

meta = pd.read_csv(f"{DRIVE_ROOT}/metadata.csv", low_memory=False)
meta[PROVINCE_COL] = meta[PROVINCE_COL].astype(str)
CLASSES = sorted(meta[PROVINCE_COL].unique()); C = len(CLASSES)
y = meta[PROVINCE_COL].map({c: i for i, c in enumerate(CLASSES)}).values
use = ~meta["exclude"].astype(bool).values
tr, va, te = [(meta["split"].values == s) & use for s in ("train", "valid", "test")]
region_of = np.array([meta.loc[meta[PROVINCE_COL] == c, "region"].mode().iloc[0] for c in CLASSES])
GROUPS = {s: (g.index.values, g["row"].values) for s, g in meta.groupby("shard")}
print(f"{C} tỉnh | train {tr.sum()} | valid {va.sum()} | test {te.sum()}")

RESULTS, PREDS, EXTRA = {}, {}, {}
BS_IDX = np.random.default_rng(SEED).integers(0, te.sum(), (1000, te.sum()))

def evaluate(name, scores):
    scores = np.asarray(scores, dtype=np.float32)
    pred, yt = scores.argmax(1), y[te]
    ok = pred == yt
    lo, hi = np.percentile(ok[BS_IDX].mean(1), [2.5, 97.5])
    r = dict(acc=ok.mean(), acc_ci95=[lo, hi], macro_f1=f1_score(yt, pred, average="macro"),
             top5=top_k_accuracy_score(yt, scores, k=min(5, C - 1), labels=np.arange(C)),
             region_acc=np.mean(region_of[pred] == region_of[yt]))
    r = {k: (np.round(v, 4).tolist() if isinstance(v, list) else round(float(v), 4)) for k, v in r.items()}
    RESULTS[name], PREDS[name] = r, pred
    np.save(f"{RES_DIR}/pred_{name}.npy", pred)
    print(f"  >> {name}: {r}")
    return r

def fit_sklearn(make_clf, X, grid):
    """Chọn siêu tham số theo valid, trả về điểm trên test (n_test, C)."""
    best = (-1, None, None)
    for p in grid:
        clf = make_clf(p).fit(X[tr], y[tr])
        acc = (clf.predict(X[va]) == y[va]).mean()
        print(f"    {p}: valid acc {acc:.4f}")
        if acc > best[0]:
            best = (acc, p, clf)
    clf = best[2]
    s = clf.predict_proba(X[te]) if hasattr(clf, "predict_proba") else clf.decision_function(X[te])
    full = np.full((te.sum(), C), -1e9, dtype=np.float32)
    full[:, clf.classes_] = s
    return full, best[1], clf

def read_audio(shard):
    col = pq.read_table(f"{LOCAL_DATA}/{shard}", columns=["audio"]).column(0)
    return lambda r: sf.read(io.BytesIO(col[int(r)].as_py()), dtype="float32")[0]

## A. Baseline âm học thô (MFCC-stats & LTAS)
**LTAS** chỉ là phổ log-mel trung bình của cả câu, gần như không mang thông tin ngữ âm, chủ yếu phản ánh micro, phòng thu và giới tính.
Nếu LTAS đạt gần bằng XLS-R thì mô hình đang học **kênh thu**, không phải phương ngữ.

In [ ]:
def acoustic_feats(shard):
    idx, rows = GROUPS[shard]
    get = read_audio(shard)
    out = []
    for mi, r in zip(idx, rows):
        y_ = get(r)
        mel = librosa.feature.melspectrogram(y=y_, sr=16000, n_fft=400, hop_length=160, n_mels=80)
        mf = librosa.feature.mfcc(S=librosa.power_to_db(mel), n_mfcc=40)
        if mf.shape[1] >= 9:
            M = np.vstack([mf, librosa.feature.delta(mf), librosa.feature.delta(mf, order=2)])
        else:
            M = np.vstack([mf, np.zeros_like(mf), np.zeros_like(mf)])
        out.append((mi, np.concatenate([M.mean(1), M.std(1)]), np.log(mel + 1e-6).mean(1)))
    return out

if RUN["acoustic"]:
    path = f"{FEAT_DIR}/acoustic.npz"
    if os.path.exists(path) and np.load(path)["mfcc"].shape[0] == len(meta):
        z = np.load(path); MFCC, LTAS = z["mfcc"], z["ltas"]
    else:
        t = time.time()
        MFCC, LTAS = np.zeros((len(meta), 240), np.float32), np.zeros((len(meta), 80), np.float32)
        with mp.get_context("fork").Pool(WORKERS) as pool:
            for k, res in enumerate(pool.imap_unordered(acoustic_feats, list(GROUPS)), 1):
                for mi, a, b in res:
                    MFCC[mi], LTAS[mi] = a, b
                if k % 20 == 0:
                    print(f"  {k}/{len(GROUPS)} shard, {time.time() - t:.0f}s", flush=True)
        np.savez(path, mfcc=MFCC, ltas=LTAS)
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    lr = lambda c: make_pipeline(StandardScaler(), LogisticRegression(C=c, max_iter=2000))
    for name, X in [("acoustic_ltas", LTAS), ("acoustic_mfcc", MFCC)]:
        print(name)
        s, p, _ = fit_sklearn(lr, X, [0.01, 0.1, 1.0])
        evaluate(name, s)

## B. Baseline chỉ văn bản
Ngoài kết quả, ô này còn kiểm tra **rò rỉ nội dung**, tức transcript có nhắc thẳng tên tỉnh hay không, và in ra các từ đặc trưng nhất của từng tỉnh.

In [ ]:
texts = meta[TEXT_COL].fillna("").astype(str).str.lower().values

def strip_prefix(n):
    n = str(n).lower().strip()
    for p in ("thành phố ", "tp. ", "tp ", "tỉnh "):
        if n.startswith(p):
            n = n[len(p):]
    return n

names = meta["province_name"].map(strip_prefix).values
EXTRA["text_mentions_own_province_rate"] = round(float(np.mean(
    [n in t for n, t in zip(names[te], texts[te])])), 4)
print("Tỉ lệ câu test nhắc tên tỉnh của chính nó:", EXTRA["text_mentions_own_province_rate"])

if RUN["text_tfidf"]:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.svm import LinearSVC
    from scipy.sparse import hstack, csr_matrix
    vw = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    vc = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True, max_features=300_000)
    vw.fit(texts[tr]); vc.fit(texts[tr])
    Xt = csr_matrix(hstack([vw.transform(texts), vc.transform(texts)]))
    print("text_tfidf", Xt.shape)
    s, p, svm = fit_sklearn(lambda c: LinearSVC(C=c), Xt, [0.1, 0.5, 1.0])
    evaluate("text_tfidf", s)
    vocab = vw.get_feature_names_out(); nw = len(vocab)
    pname = meta.groupby(PROVINCE_COL)["province_name"].first()
    EXTRA["top_words"] = {pname[CLASSES[c]]: vocab[np.argsort(-svm.coef_[k, :nw])[:10]].tolist()
                          for k, c in enumerate(svm.classes_[:63])}
    for k in list(EXTRA["top_words"])[:8]:
        print(f"  {k}: {', '.join(EXTRA['top_words'][k])}")

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if device.type == "cuda" else None
print("Thiết bị:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "", "| AMP:", AMP)
autocast = lambda: torch.autocast(device_type=device.type, dtype=AMP or torch.bfloat16, enabled=AMP is not None)
torch.manual_seed(SEED)

In [ ]:
def segment(ts):
    try:
        from pyvi import ViTokenizer
        return [ViTokenizer.tokenize(t) for t in ts], "pyvi"
    except Exception as e:
        print("Không dùng được pyvi, dùng văn bản chưa tách từ:", repr(e)[:100])
        return list(ts), "none"

def run_phobert():
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    raw = meta[TEXT_COL].fillna("").astype(str).values
    seg, EXTRA["phobert_segmenter"] = segment(raw)
    tok = AutoTokenizer.from_pretrained(PHOBERT_MODEL)
    ids = tok(seg, truncation=True, max_length=PHOBERT_MAX_LEN)["input_ids"]
    rng = np.random.default_rng(SEED)

    def batches(idx, shuffle):
        idx = np.array(idx)
        if shuffle:
            rng.shuffle(idx)
        for i in range(0, len(idx), PHOBERT_BS):
            b = idx[i:i + PHOBERT_BS]; L = max(len(ids[j]) for j in b)
            x = np.full((len(b), L), tok.pad_token_id); m = np.zeros((len(b), L), dtype=np.int64)
            for k, j in enumerate(b):
                x[k, :len(ids[j])] = ids[j]; m[k, :len(ids[j])] = 1
            yield torch.tensor(x, device=device), torch.tensor(m, device=device), torch.tensor(y[b], device=device)

    @torch.no_grad()
    def predict(idx):
        model.eval(); out = []
        for x, m, _ in batches(idx, False):
            with autocast():
                out.append(model(input_ids=x, attention_mask=m).logits.float().softmax(-1).cpu())
        return torch.cat(out).numpy()

    model = AutoModelForSequenceClassification.from_pretrained(
        PHOBERT_MODEL, num_labels=C, ignore_mismatched_sizes=True).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=PHOBERT_LR, weight_decay=0.01)
    tr_idx, va_idx, te_idx = np.where(tr)[0], np.where(va)[0], np.where(te)[0]
    steps = PHOBERT_EPOCHS * int(np.ceil(len(tr_idx) / PHOBERT_BS))
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    scaler = torch.amp.GradScaler(device.type, enabled=AMP == torch.float16)
    best, best_state = -1, None
    for ep in range(PHOBERT_EPOCHS):
        model.train(); t = time.time()
        for x, m, yb in batches(tr_idx, True):
            with autocast():
                loss = model(input_ids=x, attention_mask=m, labels=yb).loss
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        acc = (predict(va_idx).argmax(1) == y[va_idx]).mean()
        print(f"  epoch {ep + 1}: loss {loss.item():.3f}, valid acc {acc:.4f}, {time.time() - t:.0f}s")
        if acc > best:
            best, best_state = acc, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    evaluate("text_phobert", predict(te_idx))
    del model, opt; torch.cuda.empty_cache()

if RUN["phobert"]:
    run_phobert()

## C. XLS-R đóng băng: trích đặc trưng mean-pool của 25 tầng (cache trên Drive theo shard)

In [ ]:
def save_np(path, arr):
    with open(path + ".part", "wb") as f:
        np.save(f, arr)
    os.replace(path + ".part", path)

def extract_xlsr():
    fdir = f"{FEAT_DIR}/xlsr_{XLSR_MODEL.split('/')[-1]}_{int(XLSR_MAX_SEC)}s"
    os.makedirs(fdir, exist_ok=True)
    todo = [s for s in GROUPS if not os.path.exists(f"{fdir}/{s}.npy")]
    print(f"XLS-R: {len(GROUPS) - len(todo)}/{len(GROUPS)} shard đã có cache")
    if todo:
        from transformers import AutoFeatureExtractor, AutoModel
        fe = AutoFeatureExtractor.from_pretrained(XLSR_MODEL)
        model = AutoModel.from_pretrained(XLSR_MODEL).to(device).eval()
        use_mask = getattr(fe, "return_attention_mask", False)
        t, n_done = time.time(), 0
        for k, s in enumerate(todo, 1):
            idx, rows = GROUPS[s]; get = read_audio(s)
            waves = [get(r)[: int(XLSR_MAX_SEC * 16000)] for r in rows]
            order = np.argsort([-len(w) for w in waves])
            feats = None
            for i in range(0, len(order), XLSR_BATCH):
                b = order[i:i + XLSR_BATCH]; batch = [waves[j] for j in b]
                inp = fe(batch, sampling_rate=16000, padding=True, return_attention_mask=True, return_tensors="pt")
                with torch.no_grad(), autocast():
                    hs = model(inp.input_values.to(device),
                               attention_mask=inp.attention_mask.to(device) if use_mask else None,
                               output_hidden_states=True).hidden_states
                flen = model._get_feat_extract_output_lengths(torch.tensor([len(w) for w in batch])).to(device)
                fm = (torch.arange(hs[0].shape[1], device=device)[None] < flen[:, None]).float()
                pooled = torch.stack([(h.float() * fm[..., None]).sum(1) / fm.sum(1, keepdim=True) for h in hs], 1)
                if feats is None:
                    feats = np.zeros((len(rows),) + tuple(pooled.shape[1:]), np.float16)
                feats[b] = pooled.cpu().numpy().astype(np.float16)
            save_np(f"{fdir}/{s}.npy", feats); n_done += len(rows)
            if k % 5 == 0 or k == len(todo):
                el = time.time() - t
                print(f"  {k}/{len(todo)} shard | {el / 60:.1f} phút | còn ~{el / n_done * (len(meta) * len(todo) / len(GROUPS) - n_done) / 60:.0f} phút", flush=True)
        del model; torch.cuda.empty_cache()
    first = np.load(f"{fdir}/{next(iter(GROUPS))}.npy")
    X = np.zeros((len(meta),) + first.shape[1:], np.float16)
    for s, (idx, _) in GROUPS.items():
        X[idx] = np.load(f"{fdir}/{s}.npy")
    return X

if RUN["xlsr"]:
    XL = extract_xlsr()
    print("Đặc trưng XLS-R:", XL.shape)

## C2. Probe trên GPU: từng tầng (linear) và weighted-sum các tầng (linear / MLP)
Tầng tốt nhất được chọn theo **valid**, không theo test, để tránh "nhìn trộm" tập test.

In [ ]:
class LayerLinear(nn.Module):
    def __init__(s, layer, d):
        super().__init__(); s.layer, s.fc = layer, nn.Linear(d, C)
    def forward(s, x):
        return s.fc(x[:, s.layer])

class WeightedSum(nn.Module):
    def __init__(s, n_layers, d, mlp):
        super().__init__(); s.w = nn.Parameter(torch.zeros(n_layers))
        s.head = nn.Sequential(nn.Linear(d, 512), nn.GELU(), nn.Dropout(0.3), nn.Linear(512, C)) if mlp else nn.Linear(d, C)
    def forward(s, x):
        return s.head((x.float() * s.w.softmax(0)[None, :, None]).sum(1))

@torch.no_grad()
def predict_head(head, X, bs=2048):
    head.eval()
    return torch.cat([head(X[i:i + bs].float()).softmax(-1) for i in range(0, len(X), bs)]).cpu().numpy()

def train_head(head, Xtr, ytr, Xva, yva, epochs=80, lr=1e-3, wd=1e-3, bs=256, patience=10):
    head = head.to(device); opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=wd)
    best, state, bad = -1, None, 0
    for ep in range(epochs):
        head.train(); perm = torch.randperm(len(Xtr), device=device)
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            loss = F.cross_entropy(head(Xtr[b].float()), ytr[b])
            opt.zero_grad(); loss.backward(); opt.step()
        acc = (predict_head(head, Xva).argmax(1) == yva).mean()
        if acc > best:
            best, state, bad = acc, {k: v.clone() for k, v in head.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience:
                break
    head.load_state_dict(state)
    return head, float(best)

if RUN["xlsr"]:
    Xg = torch.tensor(XL, device=device)                       # (N, L, D) fp16
    mu = Xg[torch.tensor(tr, device=device)].float().mean(0)
    sd = Xg[torch.tensor(tr, device=device)].float().std(0) + 1e-5
    for i in range(0, len(Xg), 2048):
        Xg[i:i + 2048] = ((Xg[i:i + 2048].float() - mu) / sd).half()
    sel = lambda m: Xg[torch.tensor(m, device=device)]
    Xtr_, Xva_, Xte_ = sel(tr), sel(va), sel(te)
    ytr_ = torch.tensor(y[tr], device=device)
    nL, D = Xg.shape[1], Xg.shape[2]

    layer_rows = []
    for l in range(nL):
        torch.manual_seed(SEED)
        h, va_acc = train_head(LayerLinear(l, D), Xtr_, ytr_, Xva_, y[va])
        te_s = predict_head(h, Xte_)
        layer_rows.append(dict(layer=l, valid_acc=round(va_acc, 4), test_acc=round(float((te_s.argmax(1) == y[te]).mean()), 4)))
        print(layer_rows[-1], flush=True)
        if l == 0 or va_acc > max(r["valid_acc"] for r in layer_rows[:-1]):
            best_layer, best_scores = l, te_s
    EXTRA["xlsr_layer_probe"] = layer_rows
    print(f"Tầng tốt nhất theo valid: {best_layer}")
    evaluate(f"xlsr_best_layer_linear", best_scores); EXTRA["xlsr_best_layer"] = best_layer

    for mlp in (False, True):
        torch.manual_seed(SEED)
        name = "xlsr_wsum_mlp" if mlp else "xlsr_wsum_linear"
        h, _ = train_head(WeightedSum(nL, D, mlp), Xtr_, ytr_, Xva_, y[va])
        evaluate(name, predict_head(h, Xte_))
        EXTRA[f"{name}_layer_weights"] = h.w.softmax(0).detach().cpu().numpy().round(4).tolist()
        print("  trọng số các tầng:", EXTRA[f"{name}_layer_weights"])

## Tổng hợp và so sánh cặp (paired bootstrap trên tập test)

In [ ]:
def paired(a, b):
    d = (PREDS[a] == y[te]).astype(float) - (PREDS[b] == y[te]).astype(float)
    lo, hi = np.percentile(d[BS_IDX].mean(1), [2.5, 97.5])
    return dict(diff=round(float(d.mean()), 4), ci95=[round(lo, 4), round(hi, 4)])

ref = "xlsr_wsum_mlp" if "xlsr_wsum_mlp" in PREDS else None
COMPARE = {f"{ref} - {b}": paired(ref, b) for b in PREDS if ref and b != ref}
table = pd.DataFrame(RESULTS).T.sort_values("acc", ascending=False)
print(table.to_string())
print(json.dumps(COMPARE, indent=1))
if "acoustic_ltas" in RESULTS and ref:
    ratio = RESULTS["acoustic_ltas"]["acc"] / RESULTS[ref]["acc"]
    print(f"\nLTAS / XLS-R = {ratio:.2f}  (càng gần 1 càng đáng lo về nhiễu kênh thu)")

report = dict(results=RESULTS, compare=COMPARE, extra=EXTRA, chance=round(1 / C, 4),
              n=dict(train=int(tr.sum()), valid=int(va.sum()), test=int(te.sum())),
              config=dict(xlsr=XLSR_MODEL, xlsr_max_sec=XLSR_MAX_SEC, phobert=PHOBERT_MODEL, epochs=PHOBERT_EPOCHS),
              gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
              total_minutes=round((time.time() - T0) / 60, 1))
json.dump(report, open(f"{RES_DIR}/baselines_report.json", "w"), ensure_ascii=False, indent=1, default=str)
print("\nĐã lưu", f"{RES_DIR}/baselines_report.json", "– gửi file này cho Claude.")